# CS 498 — Effective Context Utilization (v2)
### Four RoPE-aware methods replacing the naive chunk-deletion / 1-D mask baselines

| Method | Approach | RoPE-safe? | Cost |
|--------|----------|------------|------|
| **A** Substitution ablation | Replace chunk with neutral filler of *same length* | ✅ positions unchanged | O(N_chunks) passes |
| **B** InputXGrad | Gradient × embedding norm — no tokens modified | ✅ no modification | **O(1) passes** |
| **C** Attention aggregation | Read attention weights during a single forward pass | ✅ no modification | **O(1) passes** |
| **D** 4-D mask ablation | Block attention *to* chunk via proper 4-D causal mask | ✅ RoPE applied before mask | O(N_chunks) passes |

> **Why the old code was wrong**  
> `chunk_deletion_baseline.py` — deleting tokens shifts every subsequent position index,  
> so RoPE angles change for all downstream tokens (invalid measurement).  
> `attention_mask_ablation.py` — passed a 1-D padding mask; Qwen2 still applied RoPE to  
> those positions before masking, AND the 1-D mask expands differently from a  
> causal-block 4-D mask (wrong semantics).

In [ ]:
!nvidia-smi

In [ ]:
!free -h

In [ ]:
# ── 1. Install dependencies ───────────────────────────────────────────────
# The conda base has torch+cu130 pre-installed; --force-reinstall overwrites it.
# IMPORTANT: after this cell completes, restart the kernel before continuing.
import sys
!{sys.executable} -m pip install -q --force-reinstall \
    --index-url https://download.pytorch.org/whl/cu128 \
    torch
!{sys.executable} -m pip install -q \
    transformers accelerate sentence-transformers \
    numpy matplotlib pandas scipy huggingface_hub
print("Done. RESTART THE KERNEL NOW, then run from cell 2 onward.")

In [ ]:
# ── 2. GPU check ──────────────────────────────────────────────────────────
import subprocess, torch

out = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
    capture_output=True, text=True
)
print('GPU    :', out.stdout.strip() or 'NOT FOUND — set Runtime to T4 GPU')
print('PyTorch:', torch.__version__)
print('CUDA   :', torch.cuda.is_available())
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info(0)
    print(f'VRAM   : {total/1e9:.1f} GB total  {free/1e9:.1f} GB free')

if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── 3. Configuration ──────────────────────────────────────────────────────
import os

MODEL_ID         = "Qwen/Qwen2.5-1.5B-Instruct"
DTYPE            = "float16"
CHUNK_SIZE       = 128          # tokens per ablation chunk
MAX_NEW_TOKENS   = 64
CONTEXT_LENGTHS  = [512, 1024, 2048]
# Needle depths as fraction of haystack (0.1 = near start, 0.9 = near end)
NEEDLE_DEPTHS    = [0.1, 0.25, 0.5, 0.75, 0.9]
NUM_EXAMPLES     = 2            # per (length × depth) config
OUTPUT_DIR       = 'results_v2'
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'

# Methods to run — comment out any you want to skip
METHODS = [
    'substitution',   # Method A: in-place filler substitution
    'inputxgrad',     # Method B: gradient × embedding importance
    'attn_agg',       # Method C: attention weight aggregation
    'mask4d',         # Method D: proper 4-D causal-block mask
]

print('Config ready.')
print(f'Total prompt configs: {len(CONTEXT_LENGTHS)} lengths × {len(NEEDLE_DEPTHS)} depths × {NUM_EXAMPLES} examples'
      f' = {len(CONTEXT_LENGTHS)*len(NEEDLE_DEPTHS)*NUM_EXAMPLES} prompts')

In [ ]:
# ── 4. HuggingFace login (only needed if model is gated) ─────────────────
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# FIX 1 — reload model in bfloat16
# ═══════════════════════════════════════════════════════════════════════════
# Qwen2.5 was pre-trained and fine-tuned with bfloat16.
# float16 has a 5-bit exponent vs bfloat16's 8-bit exponent — it overflows
# in the RMSNorm / attention softmax of deeper layers when run with eager
# (non-fused) kernels, producing NaN hidden states and garbage logits.
# bfloat16 has the same dynamic range as float32 and never overflows here.

import torch, gc
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
print(f'Tokenizer loaded — vocab size: {len(tokenizer)}')

# Use bfloat16 if the GPU supports it (all T4/A100/L4 do since CUDA 11.0)
LOAD_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f'dtype : {LOAD_DTYPE}')

# Release old model if it exists
try:
    del model
    torch.cuda.empty_cache(); gc.collect()
except NameError:
    pass

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype              = LOAD_DTYPE,
    device_map               = 'auto',
    attn_implementation      = 'eager',   # needed for output_attentions + 4-D hooks
)
model.eval()
print(f'Model reloaded  — param dtype: {next(model.parameters()).dtype}')
if torch.cuda.is_available():
    print(f'VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# FIX 2 — correct eos_token_id and clean GENERATE_KWARGS
# ═══════════════════════════════════════════════════════════════════════════
# Qwen2.5-Instruct stops cleanly at <|im_end|> (id 151645) or
# <|endoftext|> (id 151643).  Without eos_token_id the model runs all the
# way to max_new_tokens even after it has emitted the stop token, then
# skip_special_tokens strips it and you get garbage trailing tokens.

# Pad token: keep existing (id=151643) — do NOT resize embeddings.
# We already confirmed pad_token_id != eos_token_id above.
print(f'pad_token_id : {tokenizer.pad_token_id}  ({tokenizer.pad_token})')
print(f'eos_token_id : {tokenizer.eos_token_id}  ({tokenizer.eos_token})')

# Collect all valid stop ids
_stop_ids = []
for tok in ('<|im_end|>', '<|endoftext|>'):
    tid = tokenizer.convert_tokens_to_ids(tok)
    if tid is not None and tid != tokenizer.unk_token_id:
        _stop_ids.append(tid)
if tokenizer.eos_token_id not in _stop_ids:
    _stop_ids.append(tokenizer.eos_token_id)
EOS_IDS = sorted(set(_stop_ids))
print(f'eos ids used : {EOS_IDS}')

GENERATE_KWARGS = dict(
    max_new_tokens = MAX_NEW_TOKENS,
    do_sample      = False,          # greedy — top_p/top_k/temp not needed
    pad_token_id   = tokenizer.pad_token_id,
    eos_token_id   = EOS_IDS,        # ← FIX 2: stop at im_end or endoftext
)


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# FIX 3+4 — encode_prompt always returns attention_mask;
#            safe_generate always passes it
# ═══════════════════════════════════════════════════════════════════════════
# Without an explicit attention_mask, HuggingFace tries to infer one from
# pad_token_id.  When pad_token_id==eos_token_id (common in Qwen configs)
# it can't, raises a warning, and may silently left-shift token positions.

def encode_prompt(prompt_text: str, device=DEVICE):
    """
    Wrap prompt_text in Qwen2.5-Instruct chat template and tokenise.
    Returns (input_ids, attention_mask) both shaped [1, seq_len].
    """
    messages = [
        {"role": "system",  "content": (
            "You are a helpful assistant. "
            "Answer questions using only the provided text. "
            "Be concise and exact.")},
        {"role": "user",    "content": prompt_text},
    ]
    templated = tokenizer.apply_chat_template(
        messages,
        tokenize              = False,
        add_generation_prompt = True,
    )
    enc = tokenizer(
        templated,
        return_tensors     = 'pt',
        padding            = False,
        add_special_tokens = False,  # template already contains them
    )
    return enc['input_ids'].to(device), enc['attention_mask'].to(device)


def safe_generate(input_ids, attention_mask=None, extra_kwargs=None):
    """
    Drop-in replacement for model.generate() that:
      • always passes an explicit attention_mask (all-ones if not provided)
      • uses the global GENERATE_KWARGS (with correct eos_token_id)
      • accepts extra_kwargs for per-call overrides (e.g. for 4-D mask hooks)
    """
    if attention_mask is None:
        attention_mask = torch.ones_like(input_ids)
    kw = {**GENERATE_KWARGS, **(extra_kwargs or {})}
    with torch.no_grad():
        return model.generate(input_ids, attention_mask=attention_mask, **kw)


# ── Smoke test ────────────────────────────────────────────────────────────
ids, mask = encode_prompt("The secret code is ALPHA-7. What is the secret code?")
print(f'Encoded length  : {ids.shape[1]} tokens')
print(f'Attention mask  : all 1s? {mask.all().item()}')
print(f'dtype of inputs : {ids.dtype}')

out = safe_generate(ids, mask)
ans = tokenizer.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()
print(f'Smoke-test answer : "{ans}"')
assert 'ALPHA-7' in ans, (
    f'Smoke test FAILED — got "{ans}".\n'
    'If you still see garbage: check that MODEL_ID ends in -Instruct '
    'and that bfloat16 loaded correctly above.'
)
print('✓ Smoke test passed — chat template + generation working correctly.')


In [ ]:
# ── 6. Multi-position NIAH prompt generation ─────────────────────────────
# Generates prompts inline (no external script) with full position sweep.
# Needle types: uuid (default), kv, sentence
#
# KEY DESIGN DECISION: the filler text length is calibrated so that
# tokenizer(prompt).input_ids ≈ target_length  (not just char count).

import random, string, json

HAYSTACK = (
    "The researchers published their findings after months of analysis. "
    "Several key variables were identified that influenced the outcome significantly. "
    "The team conducted follow-up experiments to verify reproducibility across sites. "
    "Equipment calibration was performed at the start of each measurement session. "
    "Statistical significance was determined using a two-tailed t-test with α = 0.05. "
)

def make_needle_uuid():
    uid = ''.join(random.choices(string.ascii_uppercase + string.digits, k=8))
    needle = f"The secret activation code is: {uid}."
    question = "What is the secret activation code?"
    return needle, question, uid

def make_needle_kv():
    val = random.randint(1000, 9999)
    key = random.choice(["system_port", "access_token", "threshold_value", "batch_index"])
    needle = f"{key} = {val}"
    question = f"What is the value of {key}?"
    return needle, question, str(val)

NEEDLE_FACTORIES = [make_needle_uuid, make_needle_kv]

def build_prompt(target_tokens, depth_frac, needle_factory=None):
    if needle_factory is None:
        needle_factory = random.choice(NEEDLE_FACTORIES)
    needle_text, question, reference = needle_factory()

    # Build haystack that reaches ~target_tokens after needle insertion
    question_suffix = f"\n\nQuestion: {question}\nAnswer (be concise):"
    q_tokens = len(tokenizer.encode(question_suffix, add_special_tokens=False))
    needle_tokens = len(tokenizer.encode(needle_text, add_special_tokens=False))
    hay_budget = target_tokens - q_tokens - needle_tokens - 10  # small slack

    # Repeat haystack until we hit budget
    hay = HAYSTACK
    while len(tokenizer.encode(hay, add_special_tokens=False)) < hay_budget:
        hay += HAYSTACK

    # Trim to exact budget
    hay_ids = tokenizer.encode(hay, add_special_tokens=False)[:hay_budget]
    hay = tokenizer.decode(hay_ids)

    # Insert needle at depth_frac of the haystack
    words = hay.split()
    insert_at = max(0, int(depth_frac * len(words)))
    words.insert(insert_at, needle_text)
    hay_with_needle = ' '.join(words)

    prompt = hay_with_needle + question_suffix
    return {
        'prompt': prompt,
        'reference': reference,
        'needle': needle_text,
        'question': question,
        'metadata': {
            'target_tokens': target_tokens,
            'depth_frac': depth_frac,
            'actual_tokens': encode_prompt(prompt)[0].shape[1],
        }
    }

# Generate all prompts
random.seed(42)
all_prompts = []
for tl in CONTEXT_LENGTHS:
    for d in NEEDLE_DEPTHS:
        for _ in range(NUM_EXAMPLES):
            all_prompts.append(build_prompt(tl, d))

print(f'Generated {len(all_prompts)} prompts')
for p in all_prompts[:4]:
    m = p['metadata']
    print(f"  target={m['target_tokens']:<6} actual={m['actual_tokens']:<6} "
          f"depth={m['depth_frac']:.2f}  ref={p['reference']}")
print('  ...')

PROMPTS_FILE = os.path.join(OUTPUT_DIR, 'prompts.jsonl')
with open(PROMPTS_FILE, 'w') as f:
    for p in all_prompts:
        f.write(json.dumps(p) + '\n')
print(f'Saved to {PROMPTS_FILE}')

In [ ]:
# ── 7. Shared utilities (metrics, EUCR, PWUP, GUD) ───────────────────────
import numpy as np

def answer_correct(generated: str, reference: str) -> bool:
    return reference.strip().lower() in generated.strip().lower()

def compute_eucr(chunk_importances: list, threshold_fracs=(0.01, 0.05, 0.10)):
    """
    Effective Utilization of Context Ratio.
    EUCR@λ = fraction of chunks whose normalised importance exceeds λ.
    """
    arr = np.array(chunk_importances, dtype=float)
    total = arr.sum()
    if total == 0:
        return {str(lam): 0.0 for lam in threshold_fracs}
    norm = arr / total
    return {f'{lam:.2f}': float((norm > lam).mean()) for lam in threshold_fracs}

def compute_pwup(chunk_importances: list):
    """
    Position-Weighted Utilization Profile.
    Splits context into B(eginning), M(iddle), E(nd) thirds and returns
    the mean normalised importance for each region.
    """
    arr = np.array(chunk_importances, dtype=float)
    n = len(arr)
    total = arr.sum()
    if total == 0:
        return {'B': 0.0, 'M': 0.0, 'E': 0.0}
    norm = arr / total
    t1, t2 = n // 3, 2 * n // 3
    return {
        'B': float(norm[:t1].mean()),
        'M': float(norm[t1:t2].mean()),
        'E': float(norm[t2:].mean()),
    }

def compute_gud(chunk_importances: list):
    """
    Gradient Utilization Decay — Spearman correlation between chunk index
    and importance. Negative value = recency bias (later chunks dominate).
    """
    from scipy.stats import spearmanr
    arr = np.array(chunk_importances, dtype=float)
    if len(arr) < 3 or arr.std() == 0:
        return float('nan')
    r, _ = spearmanr(np.arange(len(arr)), arr)
    return float(r)

def needle_chunk_rank(chunk_importances: list, needle_chunk_idx: int):
    """
    Rank of the needle chunk by descending importance (1 = most important).
    Lower is better (model correctly prioritises the needle).
    """
    arr = np.array(chunk_importances, dtype=float)
    rank = int((-arr).argsort().tolist().index(needle_chunk_idx)) + 1
    return rank

print('Utility functions defined.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# FIX 3 (continued) — substitution ablation with a neutral content filler
# ═══════════════════════════════════════════════════════════════════════════
# Old code used pad_token_id (151643 = <|endoftext|>) as filler.
# The model recognises this as end-of-document → hidden states collapse.
# We need a high-frequency, semantically neutral CONTENT token.
# We pick the token for "." (period) — present everywhere in pre-training,
# never a special token, and its embedding is well-behaved.

_FILLER_ID = tokenizer.encode('.', add_special_tokens=False)[0]
print(f'Substitution filler: id={_FILLER_ID}  '
      f'decoded="{tokenizer.decode([_FILLER_ID])}"')


def substitution_ablation(prompt_record, chunk_size=CHUNK_SIZE):
    prompt    = prompt_record['prompt']
    reference = prompt_record['reference']
    needle    = prompt_record['needle']

    ids, mask = encode_prompt(prompt)       # FIX 4: always get attention_mask
    seq_len   = ids.shape[1]

    # Locate needle chunk
    needle_tok_ids = tokenizer.encode(needle, add_special_tokens=False)
    ids_list       = ids[0].tolist()
    needle_start   = next(
        (i for i in range(len(ids_list) - len(needle_tok_ids) + 1)
         if ids_list[i:i+len(needle_tok_ids)] == needle_tok_ids),
        seq_len // 2
    )
    needle_chunk_idx = needle_start // chunk_size
    num_chunks       = (seq_len + chunk_size - 1) // chunk_size

    # Baseline
    base_out      = safe_generate(ids, mask)            # FIX 4: pass mask
    baseline_text = tokenizer.decode(base_out[0, seq_len:],
                                     skip_special_tokens=True).strip()
    baseline_ok   = answer_correct(baseline_text, reference)

    chunk_importances = []
    results = []

    for ci in range(num_chunks):
        s = ci * chunk_size
        e = min(s + chunk_size, seq_len)
        ablated = ids.clone()
        ablated[0, s:e] = _FILLER_ID          # FIX 3: neutral '.' filler
        # attention_mask unchanged — all positions still visible

        out = safe_generate(ablated, mask)     # FIX 4: pass mask
        gen = tokenizer.decode(out[0, seq_len:],
                               skip_special_tokens=True).strip()
        correct = answer_correct(gen, reference)
        changed = gen != baseline_text
        imp = 1.0 if (baseline_ok and not correct) else (0.5 if changed else 0.0)
        chunk_importances.append(imp)
        results.append({'chunk_idx': ci, 'chunk_start': s, 'chunk_end': e,
                        'output': gen, 'correct': correct, 'answer_changed': changed})

    return {
        'method': 'substitution',
        'baseline_text': baseline_text, 'baseline_correct': baseline_ok,
        'chunk_importances': chunk_importances,
        'needle_chunk_idx': needle_chunk_idx,
        'eucr': compute_eucr(chunk_importances),
        'pwup': compute_pwup(chunk_importances),
        'gud':  compute_gud(chunk_importances),
        'needle_rank': needle_chunk_rank(chunk_importances, needle_chunk_idx),
        'ablation_results': results, 'metadata': prompt_record['metadata'],
    }

print('Method A (fixed) defined.')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Method B (fixed) — InputXGrad with chat template + attention_mask
# ═══════════════════════════════════════════════════════════════════════════

def inputxgrad_importance(prompt_record, chunk_size=CHUNK_SIZE):
    prompt    = prompt_record['prompt']
    reference = prompt_record['reference']
    needle    = prompt_record['needle']

    ids, mask = encode_prompt(prompt)
    seq_len   = ids.shape[1]

    # Greedy decode to get the model's own answer tokens
    base_out  = safe_generate(ids, mask)
    answer_ids = base_out[0, seq_len:]
    full_ids   = torch.cat([ids[0], answer_ids]).unsqueeze(0)
    # attention mask for full sequence (prompt + answer, all attended)
    full_mask  = torch.ones(1, full_ids.shape[1],
                            dtype=torch.long, device=DEVICE)

    needle_tok_ids = tokenizer.encode(needle, add_special_tokens=False)
    ids_list       = ids[0].tolist()
    needle_start   = next(
        (i for i in range(len(ids_list) - len(needle_tok_ids) + 1)
         if ids_list[i:i+len(needle_tok_ids)] == needle_tok_ids),
        seq_len // 2
    )
    needle_chunk_idx = needle_start // chunk_size

    # Forward through embedding layer — requires gradient
    embed_layer = model.get_input_embeddings()
    embeds = embed_layer(full_ids).detach().requires_grad_(True)

    # We must also build a proper causal attention_mask for the full sequence
    # shaped [1, full_len] so the model knows all positions are real tokens
    outputs = model(
        inputs_embeds  = embeds,
        attention_mask = full_mask,   # FIX 4
    )
    logits = outputs.logits   # [1, full_len, vocab]

    # Loss = -log p(answer_tokens | prompt)
    answer_logits = logits[0, seq_len - 1:-1, :]   # shifted
    loss = torch.nn.functional.cross_entropy(answer_logits, answer_ids)
    loss.backward()

    with torch.no_grad():
        importance = (embeds.grad[0, :seq_len] * embeds[0, :seq_len]).norm(dim=-1)
        importance = importance.cpu().float().numpy()

    num_chunks = (seq_len + chunk_size - 1) // chunk_size
    chunk_importances = [
        float(importance[ci*chunk_size : min((ci+1)*chunk_size, seq_len)].mean())
        for ci in range(num_chunks)
    ]

    baseline_text    = tokenizer.decode(answer_ids, skip_special_tokens=True).strip()
    baseline_correct = answer_correct(baseline_text, reference)

    return {
        'method': 'inputxgrad',
        'baseline_text': baseline_text, 'baseline_correct': baseline_correct,
        'chunk_importances': chunk_importances,
        'token_importances': importance.tolist(),
        'needle_chunk_idx': needle_chunk_idx,
        'eucr': compute_eucr(chunk_importances),
        'pwup': compute_pwup(chunk_importances),
        'gud':  compute_gud(chunk_importances),
        'needle_rank': needle_chunk_rank(chunk_importances, needle_chunk_idx),
        'metadata': prompt_record['metadata'],
    }

print('Method B (fixed) defined.')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Method C (fixed) — Attention aggregation with chat template + mask
# ═══════════════════════════════════════════════════════════════════════════

def attention_aggregation(prompt_record, chunk_size=CHUNK_SIZE,
                           use_rollout=False):
    prompt    = prompt_record['prompt']
    reference = prompt_record['reference']
    needle    = prompt_record['needle']
    question  = prompt_record['question']

    ids, mask = encode_prompt(prompt)
    seq_len   = ids.shape[1]

    # Question token span inside the encoded (chat-templated) sequence.
    # We approximate: encode just the question and assume it appears at the end.
    q_toks  = tokenizer.encode(question, add_special_tokens=False)
    q_len   = len(q_toks)
    q_start = max(0, seq_len - q_len - 5)   # -5 slack for template suffix tokens
    ctx_len = q_start

    needle_tok_ids = tokenizer.encode(needle, add_special_tokens=False)
    ids_list       = ids[0].tolist()
    needle_start   = next(
        (i for i in range(len(ids_list) - len(needle_tok_ids) + 1)
         if ids_list[i:i+len(needle_tok_ids)] == needle_tok_ids),
        ctx_len // 2
    )
    needle_chunk_idx = needle_start // chunk_size

    # Single forward pass, extract attention weights
    with torch.no_grad():
        outputs = model(
            ids,
            attention_mask  = mask,    # FIX 4
            output_attentions = True,
        )

    # Stack: [n_layers, n_heads, seq_len, seq_len]
    all_attn = torch.stack(
        [a.squeeze(0).cpu().float() for a in outputs.attentions]
    )

    if not use_rollout:
        # Mean attention from question positions → context positions
        attn_q_to_ctx = all_attn[:, :, q_start:seq_len, :ctx_len]
        attn_to_ctx   = attn_q_to_ctx.mean(dim=(0, 1, 2)).numpy()
    else:
        # Attention Rollout (Abnar & Zuidema 2020)
        T = seq_len
        rollout = torch.eye(T)
        for l in range(all_attn.shape[0]):
            a = all_attn[l].mean(dim=0)
            a = a + torch.eye(T)
            a = a / (a.sum(dim=-1, keepdim=True) + 1e-9)
            rollout = a @ rollout
        attn_to_ctx = rollout[q_start:seq_len, :ctx_len].mean(dim=0).numpy()

    num_chunks = (ctx_len + chunk_size - 1) // chunk_size
    chunk_attn = [
        float(attn_to_ctx[ci*chunk_size : min((ci+1)*chunk_size, ctx_len)].mean())
        for ci in range(num_chunks)
    ]

    base_out         = safe_generate(ids, mask)
    baseline_text    = tokenizer.decode(base_out[0, seq_len:],
                                        skip_special_tokens=True).strip()
    baseline_correct = answer_correct(baseline_text, reference)

    return {
        'method': 'attn_agg' + ('_rollout' if use_rollout else ''),
        'baseline_text': baseline_text, 'baseline_correct': baseline_correct,
        'chunk_importances': chunk_attn,
        'needle_chunk_idx': needle_chunk_idx,
        'eucr': compute_eucr(chunk_attn),
        'pwup': compute_pwup(chunk_attn),
        'gud':  compute_gud(chunk_attn),
        'needle_rank': needle_chunk_rank(chunk_attn, needle_chunk_idx),
        'metadata': prompt_record['metadata'],
    }

print('Method C (fixed) defined.')


In [ ]:
# ── Method D (fixed): 4-D mask — prefill-then-decode, no hooks ───────────
#
# Root cause of the crash:
#   1. In this transformers version Qwen2Attention.forward() receives
#      hidden_states as a **keyword** argument, so `args` is empty and
#      `args[0]` throws IndexError.
#   2. model.generate() rebuilds attention_mask every decode step for the
#      KV-cache phase anyway, so hook injection via generate() was never
#      going to work correctly.
#
# Fix: pass the 4-D additive mask directly to model() (eager attn accepts it),
# capture the resulting KV cache, then run a manual greedy decode loop.
# The cache was built with chunk ci blocked → decoding from it is equivalent
# to "answering having never seen chunk ci", with zero positional side-effects.

import torch

def mask4d_ablation(prompt_record, chunk_size=CHUNK_SIZE,
                    max_new_tokens=MAX_NEW_TOKENS):
    prompt    = prompt_record['prompt']
    reference = prompt_record['reference']
    needle    = prompt_record['needle']

    input_ids, attn_mask = encode_prompt(prompt)
    seq_len   = input_ids.shape[1]
    dtype     = next(model.parameters()).dtype

    # ── Needle chunk index ────────────────────────────────────────────────
    needle_ids   = tokenizer.encode(needle, add_special_tokens=False)
    ids_list     = input_ids[0].tolist()
    needle_start = next(
        (i for i in range(len(ids_list) - len(needle_ids) + 1)
         if ids_list[i:i+len(needle_ids)] == needle_ids),
        seq_len // 2,
    )
    needle_chunk_idx = needle_start // chunk_size
    num_chunks = (seq_len + chunk_size - 1) // chunk_size

    # ── Baseline (standard greedy decode, no masking) ─────────────────────
    base_out         = safe_generate(input_ids, attn_mask)
    baseline_text    = tokenizer.decode(base_out[0, seq_len:], skip_special_tokens=True).strip()
    baseline_correct = answer_correct(baseline_text, reference)

    chunk_importances = []
    results           = []

    for ci in range(num_chunks):
        s = ci * chunk_size
        e = min(s + chunk_size, seq_len)

        # ── 4-D additive mask: causal lower-tri with columns [s:e] blocked ──
        # Shape [1, 1, seq_len, seq_len]; additive (0 = attend, -inf = block).
        # Qwen2 eager attention adds this directly to the QK^T matrix before
        # softmax, so RoPE has already been applied to Q and K — fully valid.
        causal  = torch.tril(torch.ones(seq_len, seq_len, dtype=torch.bool, device=DEVICE))
        causal[:, s:e] = False                        # block the chunk columns
        mask_4d = torch.zeros(1, 1, seq_len, seq_len, device=DEVICE, dtype=dtype)
        mask_4d.masked_fill_(~causal, torch.finfo(dtype).min)

        # ── Step 1: prefill with the blocked mask → capture KV cache ────────
        # model() with a 4-D attention_mask works directly under eager attn.
        with torch.no_grad():
            prefill = model(
                input_ids,
                attention_mask=mask_4d,
                use_cache=True,
                return_dict=True,
            )
        past_kv = prefill.past_key_values
        # First generated token comes from the last logit of the prefill
        cur_id  = prefill.logits[:, -1:, :].argmax(dim=-1)   # shape [1, 1]
        generated = [cur_id.item()]

        # ── Step 2: greedy decode from the blocked KV cache ─────────────────
        # The cache was built with chunk ci invisible, so the model decodes
        # as if that content never existed — no further masking needed.
        for _ in range(max_new_tokens - 1):
            if cur_id.item() == tokenizer.eos_token_id:
                break
            with torch.no_grad():
                step    = model(cur_id, past_key_values=past_kv,
                                use_cache=True, return_dict=True)
            past_kv = step.past_key_values
            cur_id  = step.logits[:, -1:, :].argmax(dim=-1)
            generated.append(cur_id.item())

        gen     = tokenizer.decode(generated, skip_special_tokens=True)
        correct = answer_correct(gen, reference)
        changed = gen.strip() != baseline_text.strip()
        imp     = 1.0 if (baseline_correct and not correct) else (0.5 if changed else 0.0)
        chunk_importances.append(imp)
        results.append({
            'chunk_idx': ci, 'chunk_start': s, 'chunk_end': e,
            'output': gen, 'correct': correct, 'answer_changed': changed,
        })

        # Free the (large) KV cache before the next chunk
        del past_kv, prefill
        torch.cuda.empty_cache()

    return {
        'method':           'mask4d',
        'baseline_text':    baseline_text,
        'baseline_correct': baseline_correct,
        'chunk_importances': chunk_importances,
        'needle_chunk_idx': needle_chunk_idx,
        'eucr':  compute_eucr(chunk_importances),
        'pwup':  compute_pwup(chunk_importances),
        'gud':   compute_gud(chunk_importances),
        'needle_rank': needle_chunk_rank(chunk_importances, needle_chunk_idx),
        'ablation_results': results,
        'metadata': prompt_record['metadata'],
    }

# Re-register dispatch and re-run sanity check
METHOD_FN = {
    'substitution': substitution_ablation,
    'inputxgrad':   inputxgrad_importance,
    'attn_agg':     attention_aggregation,
    'mask4d':       mask4d_ablation,
}

import time, gc
print('Sanity check (prompt 0) ...')
p0 = all_prompts[0]
for method in METHODS:
    t0 = time.time()
    try:
        r = METHOD_FN[method](p0)
        print(f'  [{method:14s}]  baseline_correct={r["baseline_correct"]}  '
              f'needle_rank={r.get("needle_rank","?")}  ({time.time()-t0:.1f}s)')
    except Exception:
        import traceback; traceback.print_exc()
    finally:
        torch.cuda.empty_cache(); gc.collect()

In [ ]:
# ── Run all configured methods across all prompts ─────────────────────────
# Runtime estimates on T4 for 30 prompts (2 examples × 5 depths × 3 lengths):
#   substitution : ~45 min  (O(N_chunks) generate calls)
#   inputxgrad   : ~ 5 min  (O(1) forward+backward per prompt)
#   attn_agg     : ~ 5 min  (O(1) forward pass per prompt)
#   mask4d       : ~45 min  (O(N_chunks) generate calls)
#
# To do a quick smoke-test, set NUM_EXAMPLES=1 and CONTEXT_LENGTHS=[512].
# To skip slow methods during development, comment them out in METHODS above.

import time, gc

METHOD_FN = {
    'substitution': substitution_ablation,
    'inputxgrad':   inputxgrad_importance,
    'attn_agg':     attention_aggregation,
    'mask4d':       mask4d_ablation,
}

all_results = {m: [] for m in METHODS}

for prompt_idx, p in enumerate(all_prompts):
    tl = p['metadata']['target_tokens']
    d  = p['metadata']['depth_frac']
    print(f'\n[{prompt_idx+1}/{len(all_prompts)}] len={tl} depth={d:.2f}  ref={p["reference"]}')

    for method in METHODS:
        t0 = time.time()
        try:
            res = METHOD_FN[method](p)
            res['prompt_idx'] = prompt_idx
            all_results[method].append(res)
            nr  = res.get('needle_rank', '?')
            ok  = res.get('baseline_correct', '?')
            elapsed = time.time() - t0
            print(f'  [{method:14s}]  needle_rank={nr}  baseline_correct={ok}  '
                  f'({elapsed:.1f}s)')
        except Exception as ex:
            print(f'  [{method:14s}]  ERROR: {ex}')
        finally:
            torch.cuda.empty_cache()
            gc.collect()

# Save results
results_file = os.path.join(OUTPUT_DIR, 'all_results.json')
with open(results_file, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'\nResults saved → {results_file}')

In [ ]:
# ── Metrics summary table ─────────────────────────────────────────────────
import pandas as pd
import numpy as np
from IPython.display import display

EUCR_KEY = '0.05'

rows = []
for method, results_list in all_results.items():
    for r in results_list:
        m = r.get('metadata', {})
        eucr = (r.get('eucr') or {})
        pwup = (r.get('pwup') or {})
        rows.append({
            'method':          method,
            'target_tokens':   m.get('target_tokens'),
            'depth_frac':      m.get('depth_frac'),
            'baseline_correct': r.get('baseline_correct'),
            'needle_rank':     r.get('needle_rank'),
            f'EUCR@{EUCR_KEY}':  eucr.get(EUCR_KEY, np.nan),
            'PWUP_B':          pwup.get('B', np.nan),
            'PWUP_M':          pwup.get('M', np.nan),
            'PWUP_E':          pwup.get('E', np.nan),
            'GUD':             r.get('gud', np.nan),
        })

df = pd.DataFrame(rows)

# Aggregate by method × context_length
summary = (
    df.groupby(['method', 'target_tokens'])
      .agg(
          baseline_acc  = ('baseline_correct', 'mean'),
          needle_rank   = ('needle_rank',      'mean'),
          eucr          = (f'EUCR@{EUCR_KEY}', 'mean'),
          pwup_b        = ('PWUP_B',           'mean'),
          pwup_m        = ('PWUP_M',           'mean'),
          pwup_e        = ('PWUP_E',           'mean'),
          gud           = ('GUD',              'mean'),
      )
      .round(3)
)

print(f'EUCR threshold: λ = {EUCR_KEY}\n')
display(summary)

summary_file = os.path.join(OUTPUT_DIR, 'summary.csv')
summary.to_csv(summary_file)
print(f'Summary saved → {summary_file}')

In [ ]:
# ── Visualizations ────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

plt.rcParams.update({'font.size': 10, 'axes.spines.top': False,
                     'axes.spines.right': False})

# ── Fig 1: Needle rank by depth × method ─────────────────────────────────
fig, axes = plt.subplots(1, len(METHODS), figsize=(5 * len(METHODS), 4), sharey=True)
if len(METHODS) == 1:
    axes = [axes]
for ax, method in zip(axes, METHODS):
    sub = df[df['method'] == method]
    for tl, grp in sub.groupby('target_tokens'):
        grp_sorted = grp.sort_values('depth_frac')
        ax.plot(grp_sorted['depth_frac'], grp_sorted['needle_rank'],
                marker='o', label=f'{tl} tok')
    ax.set_title(method)
    ax.set_xlabel('Needle depth (fraction)')
    ax.set_ylabel('Needle chunk rank (lower = better)')
    ax.legend(fontsize=8)
    ax.axhline(1, color='gray', linestyle='--', linewidth=0.8)
plt.suptitle('Needle chunk rank vs depth — lower = model correctly identifies needle',
             y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'needle_rank_by_depth.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── Fig 2: PWUP profiles — B/M/E bar charts per method ───────────────────
fig, axes = plt.subplots(1, len(METHODS), figsize=(4 * len(METHODS), 4))
if len(METHODS) == 1:
    axes = [axes]
regions = ['PWUP_B', 'PWUP_M', 'PWUP_E']
labels  = ['Beginning', 'Middle', 'End']
x = np.arange(3)
for ax, method in zip(axes, METHODS):
    sub = df[df['method'] == method]
    means = [sub[r].mean() for r in regions]
    bars = ax.bar(x, means, color=['#B5D4F4', '#378ADD', '#0C447C'])
    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_title(method); ax.set_ylabel('Mean normalised importance')
    ax.set_ylim(0, max(means) * 1.3 + 0.01)
    for bar, val in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)
plt.suptitle('PWUP — where in the context does each method find importance?', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'pwup_profiles.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── Fig 3: EUCR vs context length ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
for method in METHODS:
    sub = df[df['method'] == method]
    grp = sub.groupby('target_tokens')[f'EUCR@{EUCR_KEY}'].mean()
    ax.plot(grp.index, grp.values, marker='o', label=method)
ax.set_xlabel('Context length (tokens)')
ax.set_ylabel(f'EUCR @ λ={EUCR_KEY}')
ax.set_title('Effective context utilisation ratio vs context length')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'eucr_vs_length.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── Fig 4: GUD (recency/primacy bias) ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
method_means = df.groupby('method')['GUD'].mean()
colors = ['#5DCAA5' if v > 0 else '#D85A30' for v in method_means.values]
ax.bar(method_means.index, method_means.values, color=colors)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('GUD (Spearman corr: position vs importance)')
ax.set_title('GUD: positive = primacy bias, negative = recency bias')
for i, (name, val) in enumerate(method_means.items()):
    ax.text(i, val + (0.005 if val >= 0 else -0.02), f'{val:.3f}',
            ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'gud_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── Fig 5: Per-prompt token importance heatmap (InputXGrad only) ──────────
if 'inputxgrad' in all_results and all_results['inputxgrad']:
    ixg = all_results['inputxgrad']
    # Pick one example per context length for display
    shown = {}
    for r in ixg:
        tl = r['metadata']['target_tokens']
        if tl not in shown:
            shown[tl] = r
    n = len(shown)
    fig, axes = plt.subplots(n, 1, figsize=(12, 2.5 * n))
    if n == 1: axes = [axes]
    for ax, (tl, r) in zip(axes, sorted(shown.items())):
        imps = np.array(r['token_importances'])
        imps_norm = (imps - imps.min()) / (imps.max() - imps.min() + 1e-9)
        ax.imshow(imps_norm.reshape(1, -1), aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)
        ax.set_yticks([])
        ax.set_xlabel('Token position')
        ax.set_title(f'InputXGrad token importance — {tl} tokens  '
                     f'(needle chunk {r["needle_chunk_idx"]}  rank {r["needle_rank"]})')
        # Mark needle chunk
        nc  = r['needle_chunk_idx']
        s   = nc * CHUNK_SIZE
        e   = min(s + CHUNK_SIZE, len(imps))
        ax.axvline(s, color='blue', linewidth=1.5, label='needle start')
        ax.axvline(e, color='blue', linewidth=1.5, linestyle='--')
    plt.suptitle('InputXGrad: per-token importance (red = high)', y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'inputxgrad_heatmap.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

print('All figures saved to', OUTPUT_DIR)

## Interpreting the results

### What each method tells you
- **Substitution (A)** — behavioural oracle: if the answer breaks when chunk i is replaced,  
  chunk i was causally necessary. High precision, high cost.
- **InputXGrad (B)** — gradient signal: how much does the loss change if we perturb  
  the embedding of token i? Cheap, continuous, works even when the model answers  
  correctly everywhere (captures *degree* of importance, not just binary break).
- **Attention aggregation (C)** — attention as proxy: which context positions do the  
  question tokens "look at"? Note: attention ≠ importance (see Jain & Wallace 2019),  
  but correlates well with needle position for retrieval-focused prompts.
- **4-D mask (D)** — clean causal ablation: blocks the information flow FROM chunk i  
  at every layer without touching positions. Closest to the ideal counterfactual  
  "what if this text were invisible?"

### Key differences from the original implementation
| Issue | Old code | v2 |
|-------|----------|----|
| RoPE validity | ❌ token deletion shifts positions | ✅ positions never change |
| Mask semantics | ❌ 1-D padding mask | ✅ 4-D causal block mask |
| Cost | O(N_chunks) generate calls | B/C: O(1); A/D: O(N_chunks) |
| Needle position coverage | only `middle` | 5 depths (0.1→0.9) |
| Examples per config | 1 | configurable (default 2) |

### Suggested writeup framing
Compare needle_rank across methods and depths. A well-utilising model should  
show needle_rank = 1 regardless of depth. The "lost in the middle" effect  
(Liu et al. 2023) predicts rank degrades for depths near 0.5 relative to  
0.1 and 0.9 (primacy + recency bias). Your GUD metric quantifies this.

## Performance Profiling: Wall Time & Peak VRAM vs Prompt Length

Measures the **total** wall-clock time and peak GPU memory for each method
across a range of context lengths, using one prompt per length (needle at depth=0.5).

| Method | Forward passes | Expected cost scaling |
|--------|---------------|----------------------|
| substitution | O(N\_chunks + 1) | Linear with context length |
| inputxgrad   | O(1) forward + backward | Constant |
| attn\_agg    | O(1) forward pass | Constant |
| mask4d       | O(N\_chunks + 1) | Linear with context length |


In [ ]:
# ── Profiling setup ─────────────────────────────────────────────────────
import time, gc
import pandas as pd
from IPython.display import display

PROFILE_LENGTHS = [256, 512, 1024, 2048, 4096]
PROFILE_DEPTH   = 0.5   # needle in the middle — deterministic, single location

def profile_method(method_fn, prompt_record):
    """Run method_fn once and return (wall_time_s, peak_vram_mb, num_chunks)."""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    t0     = time.perf_counter()
    result = method_fn(prompt_record)
    torch.cuda.synchronize()
    wall_time    = time.perf_counter() - t0
    peak_vram_mb = torch.cuda.max_memory_allocated() / 1e6
    num_chunks   = len(result['chunk_importances'])
    return wall_time, peak_vram_mb, num_chunks

# Build one prompt per context length
random.seed(99)
profile_prompts = {
    tl: build_prompt(tl, PROFILE_DEPTH, make_needle_uuid)
    for tl in PROFILE_LENGTHS
}
print('Profile prompts (target → actual tokens):')
for tl, p in profile_prompts.items():
    print(f'  {tl} → {p["metadata"]["actual_tokens"]}')

# ── Run profiling loop ────────────────────────────────────────────────────
profile_records = []

for tl in PROFILE_LENGTHS:
    p = profile_prompts[tl]
    print(f'\n── length {tl} ─────────────────────────────────────────')
    for method in METHODS:
        try:
            wall, vram_mb, n_chunks = profile_method(METHOD_FN[method], p)
            # O(N_chunks+1) for behavioural ablation methods; O(1) for gradient/attn
            fwd = (n_chunks + 1) if method in ('substitution', 'mask4d') else 1
            profile_records.append({
                'method':        method,
                'target_tokens': tl,
                'actual_tokens': p['metadata']['actual_tokens'],
                'num_chunks':    n_chunks,
                'fwd_passes':    fwd,
                'wall_time_s':   round(wall, 3),
                'peak_vram_gb':  round(vram_mb / 1e3, 3),
            })
            print(f'  [{method:14s}]  chunks={n_chunks:<3}  fwd={fwd:<3}  '
                  f'time={wall:.2f}s  vram={vram_mb/1e3:.2f} GB')
        except Exception:
            import traceback; traceback.print_exc()
        finally:
            gc.collect()
            torch.cuda.empty_cache()

profile_df = pd.DataFrame(profile_records)
profile_csv = os.path.join(OUTPUT_DIR, 'profiling.csv')
profile_df.to_csv(profile_csv, index=False)
print(f'\nProfiling saved → {profile_csv}')
display(profile_df)


In [ ]:
# ── Profiling visualizations ─────────────────────────────────────────────
import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 10, 'axes.spines.top': False,
                     'axes.spines.right': False})

METHOD_COLORS = {
    'substitution': '#E07B39',
    'inputxgrad':   '#5DCAA5',
    'attn_agg':     '#378ADD',
    'mask4d':       '#9B59B6',
}

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# ── A: Wall time ──────────────────────────────────────────────────────────
ax = axes[0]
for method in METHODS:
    sub = profile_df[profile_df['method'] == method].sort_values('actual_tokens')
    ax.plot(sub['actual_tokens'], sub['wall_time_s'],
            marker='o', label=method, color=METHOD_COLORS.get(method))
ax.set_xlabel('Context length (tokens)')
ax.set_ylabel('Total wall time (s)')
ax.set_title('Wall time vs context length')
ax.legend(fontsize=9)

# ── B: Peak VRAM ──────────────────────────────────────────────────────────
ax = axes[1]
for method in METHODS:
    sub = profile_df[profile_df['method'] == method].sort_values('actual_tokens')
    ax.plot(sub['actual_tokens'], sub['peak_vram_gb'],
            marker='o', label=method, color=METHOD_COLORS.get(method))
ax.set_xlabel('Context length (tokens)')
ax.set_ylabel('Peak VRAM (GB)')
ax.set_title('Peak VRAM vs context length')
ax.legend(fontsize=9)

# ── C: Forward passes ─────────────────────────────────────────────────────
ax = axes[2]
for method in METHODS:
    sub = profile_df[profile_df['method'] == method].sort_values('actual_tokens')
    ax.plot(sub['actual_tokens'], sub['fwd_passes'],
            marker='o', label=method, color=METHOD_COLORS.get(method))
ax.set_xlabel('Context length (tokens)')
ax.set_ylabel('Forward passes')
ax.set_title('Forward passes vs context length')
ax.legend(fontsize=9)

plt.suptitle('Profiling: cost scaling across methods', y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'profiling_comparison.png'),
            dpi=150, bbox_inches='tight')
plt.show()

# ── Summary pivot tables ──────────────────────────────────────────────────
print('\nWall time (s) by method × context length:')
display(profile_df.pivot(index='actual_tokens', columns='method',
                          values='wall_time_s').round(2))

print('\nPeak VRAM (GB) by method × context length:')
display(profile_df.pivot(index='actual_tokens', columns='method',
                          values='peak_vram_gb').round(3))

print('\nForward passes by method × context length:')
display(profile_df.pivot(index='actual_tokens', columns='method',
                          values='fwd_passes'))

print(f'\nFigure saved → {os.path.join(OUTPUT_DIR, "profiling_comparison.png")}')
